## Simulator (fleet of uavs)

In [ ]:
from simulator import Simulator
from simulator.config import DATA_PATH, Color, Model
from simulator.entities import SimVehicle
from simulator.helpers import SimProcess, clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.planner import AutoPlan
from simulator.visualizer import (
    QGC,
    Gazebo,
    GazMarker,
    NoVisualizer,
    QGCMarker,
)

clean()

## Simulation Positions

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

d = 300
h = 20
base_homes = ENUPose.list([(0, 0, 0, 0), (0, d, 0, 180)])
base_paths = [ENU.list([(0, 0, h), (0, d, h)]), ENU.list([(0, 0, h), (0, d, h)])]


## Create Vehicles

In [ ]:
sysids = [1, 2]
colors = [Color.BLUE, Color.RED]
models = [Model.ZEPHYR, Model.ZEPHYR]
speeds = [6.0, 6.0]  # m/s
lands = [True, True]
gcs_name = f"Multicolor_{''.join([color.emoji for color in colors])}"


mission_folder = DATA_PATH / "missions"
mission_folder.mkdir(parents=True, exist_ok=True)

vehs: list[SimVehicle] = []
for sysid, base_home, base_path, color, speed, land, model in zip(
    sysids, base_homes, base_paths, colors, speeds, lands, models, strict=True
):
    mission_path = str(mission_folder / f"mission_{sysid}.waypoints")
    auto_plan = AutoPlan.from_relative_path(
        name="simple_auto_plan",
        sysid=sysid,
        gra_origin=gra_origin,
        relative_home=base_home,
        relative_path=base_path,
        land=land,
        navigation_speed=speed,
        mission_path=mission_path,
        firmware=model.firmware,
    )
    veh = SimVehicle.from_relative(
        sysid=sysid,
        gcs_name=f"{color.name}_{color.emoji}",
        plan=auto_plan,
        color=color,
        enu_origin=enu_origin,
        relative_home=base_home,
        relative_path=base_path,
        model=model,
    )
    vehs.append(veh)

## Visualizer

### Gazebo

In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_gaz = GazMarker(
    name="origin", group="origin", pos=enu_origin.unpose(), color=Color.WHITE
)
gaz.markers.append(origin_gaz)

### QGroundControl

In [ ]:
qgc = QGC(gra_origin)
origin_qgc = QGCMarker(name="origin", pos=gra_origin.unpose(), color=Color.WHITE)
qgc.markers.append(origin_qgc)

### No Visualizer

In [ ]:
novis = NoVisualizer(gra_origin)

## Simulator

In [ ]:
simulator = Simulator(
    visualizer=gaz,
    terminals=[SimProcess.GCS],
    verbose=1,
)

for veh in vehs:
    simulator.add_vehicle(veh)

simulator.show()

## Run

In [ ]:
orac = simulator.launch()
orac.run()